# Research

In [57]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_pickle("./data/telco_churn_clean.pkl")

print(df.shape)
print()
print(df.head())

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

(7043, 21)

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        C

In [58]:
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print()
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

X_train shape: (5282, 19)
X_test shape: (1761, 19)

Churn
No     0.73457
Yes    0.26543
Name: proportion, dtype: float64
Churn
No     0.73481
Yes    0.26519
Name: proportion, dtype: float64


In [59]:
num_features = X.select_dtypes(include="number").columns.tolist()
num_features.remove("SeniorCitizen")
cat_features = X.select_dtypes(exclude="number").columns.tolist()
cat_features.append("SeniorCitizen")


In [60]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, TargetEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", TargetEncoder(), cat_features),
    ]
)

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42)),
])

In [61]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [62]:
y_pred = pipeline.predict(X_test) 

y_proba = pipeline.predict_proba(X_test)[:, 1] 

In [63]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

precision = precision_score(y_test, y_pred, pos_label="Yes")
recall = recall_score(y_test, y_pred, pos_label="Yes")
f1 = f1_score(y_test, y_pred, pos_label="Yes")

roc_auc = roc_auc_score(y_test, y_proba)

baseline_metrics = {"precision": precision, "recall": recall, "f1": f1, "roc_auc": roc_auc}

---
## Baseline — анализ результатов

In [64]:
print("Baseline Metrics:\n")
for name, value in baseline_metrics.items():
    print(f"{name}: {value:.4f}")

print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}')

Baseline Metrics:

precision: 0.6503
recall: 0.4818
f1: 0.5535
roc_auc: 0.8385

Confusion Matrix:
[[1173  121]
 [ 242  225]]


In [65]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(
    y_test, y_proba, pos_label="Yes"
)

# индексы порогов, где recall >= 0.7
mask = recalls[:-1] >= 0.7  # последний элемент — для нулевого precision, отбрасываем
valid_thresholds = thresholds[mask]
valid_precisions = precisions[:-1][mask]

# выбираем тот порог, где precision максимальна
best_idx = valid_precisions.argmax()
best_threshold = valid_thresholds[best_idx]

---
## Feature Engineering (sklearn)

In [66]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer
from sklearn.pipeline import Pipeline

poly_pipeline = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler_after_poly", StandardScaler()),
])

preprocessor_fe = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", TargetEncoder(), cat_features),
        ("poly", poly_pipeline, ["tenure", "MonthlyCharges"]),
        ("bins", KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile"), ["tenure", "MonthlyCharges"]),
    ]
)

In [67]:
pipeline_fe = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("model", RandomForestClassifier(random_state=42)),
])

pipeline_fe.fit(X_train, y_train)
y_pred_fe = pipeline_fe.predict(X_test)
y_proba_fe = pipeline_fe.predict_proba(X_test)[:, 1]

/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [68]:
precision_fe = precision_score(y_test, y_pred_fe, pos_label="Yes")
recall_fe = recall_score(y_test, y_pred_fe, pos_label="Yes")
f1_fe = f1_score(y_test, y_pred_fe, pos_label="Yes")

roc_auc_fe = roc_auc_score(y_test, y_proba_fe)

fe_metrics = {"precision": precision_fe, "recall": recall_fe, "f1": f1_fe, "roc_auc": roc_auc_fe}

In [69]:
print("Feature Engineering Metrics:\n")
for name, value in fe_metrics.items():
    print(f"{name}: {value:.4f}")

print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_fe)}')

Feature Engineering Metrics:

precision: 0.6568
recall: 0.4754
f1: 0.5516
roc_auc: 0.8322

Confusion Matrix:
[[1178  116]
 [ 245  222]]


In [70]:
import json

feature_names = preprocessor_fe.get_feature_names_out().tolist()
with open("./outputs/feature_names_fe_sklearn.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print(f"Всего признаков после FE: {len(feature_names)}")
print(feature_names)

Всего признаков после FE: 26
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender', 'cat__Partner', 'cat__Dependents', 'cat__PhoneService', 'cat__MultipleLines', 'cat__InternetService', 'cat__OnlineSecurity', 'cat__OnlineBackup', 'cat__DeviceProtection', 'cat__TechSupport', 'cat__StreamingTV', 'cat__StreamingMovies', 'cat__Contract', 'cat__PaperlessBilling', 'cat__PaymentMethod', 'cat__SeniorCitizen', 'poly__tenure', 'poly__MonthlyCharges', 'poly__tenure^2', 'poly__tenure MonthlyCharges', 'poly__MonthlyCharges^2', 'bins__tenure', 'bins__MonthlyCharges']


### SFS - forward

In [71]:
import numpy as np

y_train_num = (y_train == "Yes").astype(int).to_numpy()
y_test_num = (y_test == "Yes").astype(int).to_numpy()

print(f"y_train_num: shape={y_train_num.shape}, unique={np.unique(y_train_num)}")
print(f"y_test_num:  shape={y_test_num.shape}, unique={np.unique(y_test_num)}")

y_train_num: shape=(5282,), unique=[0 1]
y_test_num:  shape=(1761,), unique=[0 1]


In [72]:
preprocessor_fe.fit(X_train, y_train)

X_train_fe = preprocessor_fe.transform(X_train)
X_test_fe = preprocessor_fe.transform(X_test)

feature_names_fe = preprocessor_fe.get_feature_names_out()

print(f"X_train_fe shape: {X_train_fe.shape}")
print(f"Всего признаков: {len(feature_names_fe)}")

X_train_fe shape: (5282, 26)
Всего признаков: 26


/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [73]:
valid_mask = np.array([
    "customerID" not in name for name in feature_names_fe
])

X_train_fe_clean = X_train_fe[:, valid_mask]
X_test_fe_clean = X_test_fe[:, valid_mask]
feature_names_clean = feature_names_fe[valid_mask]

print(f"Признаков было: {len(feature_names_fe)}")
print(f"После фильтра: {len(feature_names_clean)}")
print(f"Удалено: {feature_names_fe[~valid_mask]}")

Признаков было: 26
После фильтра: 26
Удалено: []


In [74]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf_test = RandomForestClassifier(random_state=42, n_estimators=50)
scores = cross_val_score(
    rf_test, X_train_fe_clean, y_train_num,
    scoring="f1", cv=3, n_jobs=1
)
print(f"CV f1 на всех признаках: {scores}")
print(f"CV f1 mean: {scores.mean():.4f}")

CV f1 на всех признаках: [0.56965174 0.55769231 0.5495283 ]
CV f1 mean: 0.5590


In [75]:
from sklearn.feature_selection import SequentialFeatureSelector

n_select = len(feature_names_clean) // 2

sfs = SequentialFeatureSelector(
    estimator=RandomForestClassifier(random_state=42, n_estimators=50),
    n_features_to_select=n_select,
    direction="forward",
    scoring="f1",
    cv=3,
    n_jobs=1,
)

print(f"Отбираем {n_select} из {len(feature_names_clean)} признаков...")
print("(может занять 5-15 минут)")

sfs.fit(X_train_fe_clean, y_train_num)

Отбираем 13 из 26 признаков...
(может занять 5-15 минут)


,estimator estimator: estimator instanceAn unfitted estimator.,RandomForestC...ndom_state=42)
,"n_features_to_select n_features_to_select: ""auto"", int or float, default=""auto""If `""auto""`, the behaviour depends on the `tol` parameter:- if `tol` is not `None`, then features are selected while the score change does not exceed `tol`.- otherwise, half of the features are selected.If integer, the parameter is the absolute number of features to select.If float between 0 and 1, it is the fraction of features to select... versionadded:: 1.1 The option `""auto""` was added in version 1.1... versionchanged:: 1.3 The default changed from `""warn""` to `""auto""` in 1.3.",13
,"tol tol: float, default=NoneIf the score is not incremented by at least `tol` between twoconsecutive feature additions or removals, stop adding or removing.`tol` can be negative when removing features using `direction=""backward""`.`tol` is required to be strictly positive when doing forward selection.It can be useful to reduce the number of features at the cost of a smalldecrease in the score.`tol` is enabled only when `n_features_to_select` is `""auto""`... versionadded:: 1.1",None
,"direction direction: {'forward', 'backward'}, default='forward'Whether to perform forward selection or backward selection.",'forward'
,"scoring scoring: str or callable, default=NoneScoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)`` that returns a single value. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion ` is used.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. In all othercases, :class:`~sklearn.model_selection.KFold` is used. These splittersare instantiated with `shuffle=False` so the splits will be the sameacross calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.",3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel. When evaluating a new feature toadd or remove, the cross-validation procedure is parallel over thefolds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",1
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [76]:
selected_mask = sfs.get_support()
selected_indices = sfs.get_support(indices=True)
selected_names = feature_names_clean[selected_mask]

print(f"Отобрано {len(selected_names)} признаков:")
for name in selected_names:
    print(f"  {name}")

Отобрано 13 признаков:
  cat__Dependents
  cat__PhoneService
  cat__MultipleLines
  cat__InternetService
  cat__OnlineSecurity
  cat__OnlineBackup
  cat__DeviceProtection
  cat__TechSupport
  cat__Contract
  cat__PaymentMethod
  poly__tenure^2
  bins__tenure
  bins__MonthlyCharges


In [77]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train_selected = X_train_fe_clean[:, selected_mask]
X_test_selected = X_test_fe_clean[:, selected_mask]

model_sfs = RandomForestClassifier(random_state=42, n_estimators=100)
model_sfs.fit(X_train_selected, y_train_num)

y_pred_sfs = model_sfs.predict(X_test_selected)
y_proba_sfs = model_sfs.predict_proba(X_test_selected)[:, 1]

sfs_metrics = {
    "precision": precision_score(y_test_num, y_pred_sfs),
    "recall": recall_score(y_test_num, y_pred_sfs),
    "f1": f1_score(y_test_num, y_pred_sfs),
    "roc_auc": roc_auc_score(y_test_num, y_proba_sfs),
}

for name, value in sfs_metrics.items():
    print(f"{name}: {value:.4f}")

precision: 0.5729
recall: 0.4797
f1: 0.5221
roc_auc: 0.7899


In [78]:
import json

with open("./outputs/sfs_selected_features.json", "w") as f:
    json.dump({
        "names": selected_names.tolist(),
        "indices": selected_indices.tolist(),
        "n_total": len(feature_names_clean),
        "n_selected": len(selected_names),
        "direction": "forward",
        "scoring": "f1",
    }, f, indent=2)

print("Сохранён sfs_selected_features.json")

Сохранён sfs_selected_features.json


### SFS - backward | RFE

In [79]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier

n_select = len(feature_names_clean) // 2

sfs_backward = SequentialFeatureSelector(
    estimator=RandomForestClassifier(random_state=42, n_estimators=50),
    n_features_to_select=n_select,
    direction="backward",
    scoring="f1",
    cv=3,
    n_jobs=1,
)

print(f"SFS backward: отбираем {n_select} из {len(feature_names_clean)}...")
print("(может занять 10-20 минут — backward медленнее forward)")

sfs_backward.fit(X_train_fe_clean, y_train_num)

selected_mask_backward = sfs_backward.get_support()
selected_names_backward = feature_names_clean[selected_mask_backward]

print(f"\nОтобрано {len(selected_names_backward)} признаков (backward):")
for name in selected_names_backward:
    print(f"  {name}")

SFS backward: отбираем 13 из 26...
(может занять 10-20 минут — backward медленнее forward)

Отобрано 13 признаков (backward):
  num__tenure
  num__MonthlyCharges
  num__TotalCharges
  cat__Dependents
  cat__PhoneService
  cat__MultipleLines
  cat__OnlineSecurity
  cat__OnlineBackup
  cat__TechSupport
  cat__PaperlessBilling
  cat__PaymentMethod
  cat__SeniorCitizen
  bins__MonthlyCharges


In [80]:
from sklearn.feature_selection import RFE

rfe = RFE(
    estimator=RandomForestClassifier(random_state=42, n_estimators=100),
    n_features_to_select=n_select,
    step=1,  # сколько признаков удалять за раз
)

print(f"RFE: отбираем {n_select} из {len(feature_names_clean)}...")
print("(должно быть быстрее SFS — пара минут)")

rfe.fit(X_train_fe_clean, y_train_num)

selected_mask_rfe = rfe.support_
selected_names_rfe = feature_names_clean[selected_mask_rfe]

print(f"\nОтобрано {len(selected_names_rfe)} признаков (RFE):")
for name in selected_names_rfe:
    print(f"  {name}")

RFE: отбираем 13 из 26...
(должно быть быстрее SFS — пара минут)

Отобрано 13 признаков (RFE):
  num__tenure
  num__MonthlyCharges
  num__TotalCharges
  cat__InternetService
  cat__OnlineSecurity
  cat__TechSupport
  cat__Contract
  cat__PaymentMethod
  poly__tenure
  poly__MonthlyCharges
  poly__tenure^2
  poly__tenure MonthlyCharges
  poly__MonthlyCharges^2


In [81]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# Метрики SFS backward
X_train_bw = X_train_fe_clean[:, selected_mask_backward]
X_test_bw = X_test_fe_clean[:, selected_mask_backward]
model_sfs_bw = RandomForestClassifier(random_state=42, n_estimators=100)
model_sfs_bw.fit(X_train_bw, y_train_num)
y_pred_bw = model_sfs_bw.predict(X_test_bw)
y_proba_bw = model_sfs_bw.predict_proba(X_test_bw)[:, 1]
sfs_backward_metrics = {
    "precision": precision_score(y_test_num, y_pred_bw),
    "recall": recall_score(y_test_num, y_pred_bw),
    "f1": f1_score(y_test_num, y_pred_bw),
    "roc_auc": roc_auc_score(y_test_num, y_proba_bw),
}
print('SFS backward metrics:')
for name, value in sfs_backward_metrics.items():
    print(f'  {name}: {value:.4f}')

# Метрики RFE
X_train_rfe_m = X_train_fe_clean[:, selected_mask_rfe]
X_test_rfe_m = X_test_fe_clean[:, selected_mask_rfe]
model_rfe_eval = RandomForestClassifier(random_state=42, n_estimators=100)
model_rfe_eval.fit(X_train_rfe_m, y_train_num)
y_pred_rfe_m = model_rfe_eval.predict(X_test_rfe_m)
y_proba_rfe_m = model_rfe_eval.predict_proba(X_test_rfe_m)[:, 1]
rfe_metrics = {
    "precision": precision_score(y_test_num, y_pred_rfe_m),
    "recall": recall_score(y_test_num, y_pred_rfe_m),
    "f1": f1_score(y_test_num, y_pred_rfe_m),
    "roc_auc": roc_auc_score(y_test_num, y_proba_rfe_m),
}
print('\nRFE metrics:')
for name, value in rfe_metrics.items():
    print(f'  {name}: {value:.4f}')

SFS backward metrics:
  precision: 0.6017
  recall: 0.4433
  f1: 0.5105
  roc_auc: 0.8055

RFE metrics:
  precision: 0.6022
  recall: 0.4732
  f1: 0.5300
  roc_auc: 0.8089


In [82]:
set_forward = set(selected_names)
set_backward = set(selected_names_backward)
set_rfe = set(selected_names_rfe)

print("=== Признаки, выбранные ВСЕМИ ТРЕМЯ методами ===")
in_all = set_forward & set_backward & set_rfe
for name in sorted(in_all):
    print(f"  ✓ {name}")
print(f"Всего: {len(in_all)}")

print("\n=== Только в SFS forward ===")
for name in sorted(set_forward - set_backward - set_rfe):
    print(f"  {name}")

print("\n=== Только в SFS backward ===")
for name in sorted(set_backward - set_forward - set_rfe):
    print(f"  {name}")

print("\n=== Только в RFE ===")
for name in sorted(set_rfe - set_forward - set_backward):
    print(f"  {name}")

=== Признаки, выбранные ВСЕМИ ТРЕМЯ методами ===
  ✓ cat__OnlineSecurity
  ✓ cat__PaymentMethod
  ✓ cat__TechSupport
Всего: 3

=== Только в SFS forward ===
  bins__tenure
  cat__DeviceProtection

=== Только в SFS backward ===
  cat__PaperlessBilling
  cat__SeniorCitizen

=== Только в RFE ===
  poly__MonthlyCharges
  poly__MonthlyCharges^2
  poly__tenure
  poly__tenure MonthlyCharges


In [83]:
# Если пересечение совсем маленькое — берём union или RFE-набор
final_features = list(in_all)

if len(final_features) < 5:
    print("Пересечение слишком маленькое, используем RFE-набор")
    final_features = list(set_rfe)
    final_mask = selected_mask_rfe
else:
    final_mask = np.array([name in final_features for name in feature_names_clean])

print(f"Финальный набор: {len(final_features)} признаков")
for name in sorted(final_features):
    print(f"  {name}")

Пересечение слишком маленькое, используем RFE-набор
Финальный набор: 13 признаков
  cat__Contract
  cat__InternetService
  cat__OnlineSecurity
  cat__PaymentMethod
  cat__TechSupport
  num__MonthlyCharges
  num__TotalCharges
  num__tenure
  poly__MonthlyCharges
  poly__MonthlyCharges^2
  poly__tenure
  poly__tenure MonthlyCharges
  poly__tenure^2


In [84]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train_final_fs = X_train_fe_clean[:, final_mask]
X_test_final_fs = X_test_fe_clean[:, final_mask]

model_combined_fs = RandomForestClassifier(random_state=42, n_estimators=100)
model_combined_fs.fit(X_train_final_fs, y_train_num)

y_pred_fs = model_combined_fs.predict(X_test_final_fs)
y_proba_fs = model_combined_fs.predict_proba(X_test_final_fs)[:, 1]

combined_fs_metrics = {
    "precision": precision_score(y_test_num, y_pred_fs),
    "recall": recall_score(y_test_num, y_pred_fs),
    "f1": f1_score(y_test_num, y_pred_fs),
    "roc_auc": roc_auc_score(y_test_num, y_proba_fs),
}

for name, value in combined_fs_metrics.items():
    print(f"{name}: {value:.4f}")

precision: 0.6022
recall: 0.4732
f1: 0.5300
roc_auc: 0.8089


---
## Optuna

In [85]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Чтобы Optuna не спамила INFO-логами в каждом trial'е
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [86]:
def objective(trial):
    # Пространство поиска
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    max_features = trial.suggest_float("max_features", 0.1, 1.0)

    # Собираем pipeline с этими параметрами
    pipeline_trial = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_features=max_features,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    # CV f1 на train
    scores = cross_val_score(
        pipeline_trial, X_train, y_train_num,
        scoring="f1", cv=3, n_jobs=1
    )
    return scores.mean()

In [87]:
import time

print("Запускаю Optuna, 15 trials...")

start_time = time.time()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=15)

elapsed = time.time() - start_time
print(f"\nГотово за {elapsed/60:.1f} минут")
print(f"Лучший f1 (CV на train): {study.best_value:.4f}")
print(f"Лучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Запускаю Optuna, 15 trials...

Готово за 0.9 минут
Лучший f1 (CV на train): 0.5841
Лучшие параметры:
  n_estimators: 96
  max_depth: 12
  max_features: 0.5722807884690141


In [88]:
import pandas as pd

# Все trials в виде DataFrame
trials_df = study.trials_dataframe()
print(trials_df[["number", "value", "params_n_estimators", "params_max_depth", "params_max_features"]].sort_values("value", ascending=False).head(10))

    number     value  params_n_estimators  params_max_depth  \
5        5  0.584115                   96                12   
9        9  0.579268                  179                20   
11      11  0.575011                  125                20   
8        8  0.574843                  164                25   
4        4  0.574741                  258                10   
10      10  0.574497                  102                17   
1        1  0.572870                  200                 9   
12      12  0.570591                  203                18   
14      14  0.569793                   56                15   
13      13  0.569016                  294                23   

    params_max_features  
5              0.572281  
9              0.141805  
11             0.134853  
8              0.279706  
4              0.263642  
10             0.456568  
1              0.240395  
12             0.805267  
14             0.577024  
13             0.466503  


In [89]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

best_params = study.best_params

pipeline_optuna = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        max_features=best_params["max_features"],
        random_state=42,
        n_jobs=-1,
    )),
])

pipeline_optuna.fit(X_train, y_train_num)

y_pred_opt = pipeline_optuna.predict(X_test)
y_proba_opt = pipeline_optuna.predict_proba(X_test)[:, 1]

optuna_metrics = {
    "precision": precision_score(y_test_num, y_pred_opt),
    "recall": recall_score(y_test_num, y_pred_opt),
    "f1": f1_score(y_test_num, y_pred_opt),
    "roc_auc": roc_auc_score(y_test_num, y_proba_opt),
}

print("Метрики на test:")
for name, value in optuna_metrics.items():
    print(f"  {name}: {value:.4f}")

print(f"\nДля сравнения, baseline f1: 0.5665")
print(f"Прирост: {optuna_metrics['f1'] - 0.5665:+.4f}")

Метрики на test:
  precision: 0.6300
  recall: 0.5032
  f1: 0.5595
  roc_auc: 0.8386

Для сравнения, baseline f1: 0.5665
Прирост: -0.0070


---
## AutoFeat

In [90]:
encoder_for_autofeat = ColumnTransformer(
    transformers=[
        ("cat", TargetEncoder(target_type="binary"), cat_features),
    ],
    remainder="passthrough",
)

X_train_encoded_arr = encoder_for_autofeat.fit_transform(X_train, y_train)
X_test_encoded_arr = encoder_for_autofeat.transform(X_test)

encoded_feature_names = encoder_for_autofeat.get_feature_names_out()

# autofeat ожидает DataFrame, не numpy array
X_train_encoded = pd.DataFrame(X_train_encoded_arr, columns=encoded_feature_names, index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded_arr, columns=encoded_feature_names, index=X_test.index)

print(f"Train shape: {X_train_encoded.shape}")
print(X_train_encoded.head())

Train shape: (5282, 19)
      cat__gender  cat__Partner  cat__Dependents  cat__PhoneService  \
6661     0.269613      0.197259         0.149375           0.241352   
4811     0.261100      0.327562         0.313309           0.270596   
2193     0.266948      0.328584         0.149030           0.267927   
1904     0.269824      0.327562         0.313309           0.270596   
6667     0.264035      0.328584         0.315096           0.267927   

      cat__MultipleLines  cat__InternetService  cat__OnlineSecurity  \
6661            0.241352              0.192321             0.415795   
4811            0.256862              0.182352             0.417487   
2193            0.243720              0.074619             0.074619   
1904            0.285941              0.424987             0.417487   
6667            0.243720              0.420078             0.419185   

      cat__OnlineBackup  cat__DeviceProtection  cat__TechSupport  \
6661           0.214073               0.387156        

In [91]:
from autofeat import AutoFeatClassifier

afc = AutoFeatClassifier(
    feateng_steps=1,
    featsel_runs=3,
    n_jobs=-1,
    verbose=1,
)

print("Начинаю генерацию признаков autofeat... (5-15 минут)")
X_train_autofeat = afc.fit_transform(X_train_encoded, y_train)
X_test_autofeat = afc.transform(X_test_encoded)

print(f"\nПризнаков было: {X_train_encoded.shape[1]}")
print(f"Признаков стало: {X_train_autofeat.shape[1]}")
print(f"Новые признаки:")
new_features = [c for c in X_train_autofeat.columns if c not in X_train_encoded.columns]
for f in new_features:
    print(f"  {f}")

2026-05-11 22:13:22,918 INFO: [AutoFeat] The 1 step feature engineering process could generate up to 133 features.
2026-05-11 22:13:22,925 INFO: [AutoFeat] With 5282 data points this new feature matrix would use about 0.00 gb of space.
2026-05-11 22:13:23,006 INFO: [feateng] Step 1: transformation of original features


Начинаю генерацию признаков autofeat... (5-15 минут)


2026-05-11 22:13:28,975 INFO: [feateng] Generated 107 transformed features from 19 original features - done.
2026-05-11 22:13:29,000 INFO: [feateng] Generated altogether 109 new features in 1 steps
2026-05-11 22:13:29,002 INFO: [feateng] Removing correlated features, as well as additions at the highest level
2026-05-11 22:13:29,030 INFO: [feateng] Generated a total of 10 additional features


[featsel] Scaling data...done.
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.


2026-05-11 22:13:36,435 INFO: [featsel] Feature selection run 2/3
2026-05-11 22:13:36,440 INFO: [featsel] Feature selection run 3/3
2026-05-11 22:13:36,450 INFO: [featsel] Feature selection run 1/3


...

2026-05-11 22:13:49,539 INFO: [featsel] 17 features after 3 feature selection runs
/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/autofeat/featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
2026-05-11 22:13:49,563 INFO: [featsel] 16 features after correlation filtering


[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:   20.5s finished


2026-05-11 22:13:51,724 INFO: [featsel] 16 features after noise filtering
2026-05-11 22:13:51,726 INFO: [AutoFeat] Computing 2 new features.


2026-05-11 22:13:52,261 INFO: [AutoFeat]     2/    2 new features ...done.
2026-05-11 22:13:52,266 INFO: [AutoFeat] Final dataframe with 21 feature columns (2 new).
2026-05-11 22:13:52,268 INFO: [AutoFeat] Training final classification model.
2026-05-11 22:14:00,654 INFO: [AutoFeat] Trained model: largest coefficients:
2026-05-11 22:14:00,658 INFO: [1.5804152e-22]
2026-05-11 22:14:00,670 INFO: [AutoFeat] Final score: 0.7331
2026-05-11 22:14:00,675 INFO: [AutoFeat] Computing 2 new features.
2026-05-11 22:14:00,690 INFO: [AutoFeat]     2/    2 new features ...done.


[AutoFeat]     1/    2 new features
Признаков было: 19
Признаков стало: 21
Новые признаки:
  1/cat__Contract
  remainder__TotalCharges**3


---

## Catboost

In [92]:
cat_features_idx = [X_train.columns.get_loc(col) for col in cat_features]
print(f"Категориальные индексы: {cat_features_idx}")
print([X_train.columns[i] for i in cat_features_idx])

Категориальные индексы: [0, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 1]
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']


In [93]:
from sklearn.model_selection import train_test_split

X_train_cb, X_eval_cb, y_train_cb, y_eval_cb = train_test_split(
    X_train, y_train_num,
    test_size=0.2,
    stratify=y_train_num,
    random_state=42,
)

print(f"Train для CatBoost: {X_train_cb.shape}")
print(f"Eval для early stopping: {X_eval_cb.shape}")

Train для CatBoost: (4225, 19)
Eval для early stopping: (1057, 19)


In [94]:
from catboost import CatBoostClassifier

cb_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    cat_features=cat_features_idx,
    eval_metric="F1",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100,
)

cb_model.fit(
    X_train_cb, y_train_cb,
    eval_set=(X_eval_cb, y_eval_cb),
)

print(f"Деревьев: {cb_model.tree_count_}, лучшая итерация: {cb_model.best_iteration_}")

0:	learn: 0.4065147	test: 0.4070352	best: 0.4070352 (0)	total: 128ms	remaining: 4m 15s
100:	learn: 0.6139016	test: 0.6159844	best: 0.6159844 (100)	total: 3.11s	remaining: 58.5s
200:	learn: 0.6413207	test: 0.6204239	best: 0.6254826 (161)	total: 5.32s	remaining: 47.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6254826255
bestIteration = 161

Shrink model to first 162 iterations.
Деревьев: 162, лучшая итерация: 161


In [95]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

y_pred_cb = cb_model.predict(X_test)
y_proba_cb = cb_model.predict_proba(X_test)[:, 1]

catboost_metrics = {
    "precision": precision_score(y_test_num, y_pred_cb),
    "recall": recall_score(y_test_num, y_pred_cb),
    "f1": f1_score(y_test_num, y_pred_cb),
    "roc_auc": roc_auc_score(y_test_num, y_proba_cb),
}

print("CatBoost метрики на test:")
for name, value in catboost_metrics.items():
    print(f"  {name}: {value:.4f}")

print(f"\nСравнение:")
print(f"  Baseline RF f1: 0.5665")
print(f"  Optuna RF f1:   0.5864")
print(f"  CatBoost f1:    {catboost_metrics['f1']:.4f}")

CatBoost метрики на test:
  precision: 0.6696
  recall: 0.4818
  f1: 0.5604
  roc_auc: 0.8444

Сравнение:
  Baseline RF f1: 0.5665
  Optuna RF f1:   0.5864
  CatBoost f1:    0.5604


In [96]:
import pandas as pd

feature_importance = cb_model.get_feature_importance()
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": feature_importance,
}).sort_values("importance", ascending=False)

print("Топ-10 признаков по важности:")
print(importance_df.head(10))

Топ-10 признаков по важности:
            feature  importance
14         Contract   22.017630
4            tenure   13.415637
17   MonthlyCharges   11.944844
8    OnlineSecurity    8.652397
7   InternetService    6.517686
11      TechSupport    5.460137
18     TotalCharges    5.253556
6     MultipleLines    4.496722
16    PaymentMethod    4.451898
9      OnlineBackup    4.101258


### CatBoost+Optuna

In [97]:
def objective_cb(trial):
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    depth = trial.suggest_int("depth", 4, 10)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1.0, 10.0)

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        cat_features=cat_features_idx,
        eval_metric="F1",
        early_stopping_rounds=50,
        random_seed=42,
        verbose=False,
    )

    model.fit(
        X_train_cb, y_train_cb,
        eval_set=(X_eval_cb, y_eval_cb),
    )

    # Метрика на eval (не на test!) — для честности
    y_pred_eval = model.predict(X_eval_cb)
    return f1_score(y_eval_cb, y_pred_eval)


print("Optuna для CatBoost: 15 trials...")
study_cb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study_cb.optimize(objective_cb, n_trials=15)

print(f"\nЛучший f1 (eval): {study_cb.best_value:.4f}")
print(f"Лучшие параметры: {study_cb.best_params}")

Optuna для CatBoost: 15 trials...

Лучший f1 (eval): 0.6211
Лучшие параметры: {'learning_rate': 0.07973550444866762, 'depth': 4, 'l2_leaf_reg': 3.4106078274227682}


In [98]:
from catboost import CatBoostClassifier
import pandas as pd

best_cb_params = study_cb.best_params

# склеиваем обратно — разделяли только для подбора параметров
X_train_full = pd.concat([X_train_cb, X_eval_cb])
y_train_full = pd.concat([
    pd.Series(y_train_cb, index=X_train_cb.index),
    pd.Series(y_eval_cb, index=X_eval_cb.index),
])
X_train_full = X_train_full.loc[X_train.index]
y_train_full = y_train_full.loc[X_train.index].to_numpy()

print(f"Финальный train shape: {X_train_full.shape}")

X_train_final, X_eval_final, y_train_final, y_eval_final = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15, stratify=y_train_full, random_state=42,
)

cb_model_tuned = CatBoostClassifier(
    iterations=2000,
    learning_rate=best_cb_params["learning_rate"],
    depth=best_cb_params["depth"],
    l2_leaf_reg=best_cb_params["l2_leaf_reg"],
    cat_features=cat_features_idx,
    eval_metric="F1",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100,
)

cb_model_tuned.fit(
    X_train_final, y_train_final,
    eval_set=(X_eval_final, y_eval_final),
)

print(f"Деревьев: {cb_model_tuned.tree_count_}, лучшая итерация: {cb_model_tuned.best_iteration_}")

Финальный train shape: (5282, 19)
0:	learn: 0.4111045	test: 0.4115756	best: 0.4115756 (0)	total: 87.7ms	remaining: 2m 55s
100:	learn: 0.6072607	test: 0.6256410	best: 0.6310433 (80)	total: 1.48s	remaining: 27.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.631043257
bestIteration = 80

Shrink model to first 81 iterations.
Деревьев: 81, лучшая итерация: 80


In [99]:
y_pred_cb_tuned = cb_model_tuned.predict(X_test)
y_proba_cb_tuned = cb_model_tuned.predict_proba(X_test)[:, 1]

catboost_tuned_metrics = {
    "precision": precision_score(y_test_num, y_pred_cb_tuned),
    "recall": recall_score(y_test_num, y_pred_cb_tuned),
    "f1": f1_score(y_test_num, y_pred_cb_tuned),
    "roc_auc": roc_auc_score(y_test_num, y_proba_cb_tuned),
}

print("CatBoost tuned метрики на test:")
for name, value in catboost_tuned_metrics.items():
    print(f"  {name}: {value:.4f}")

print(f"\nСравнение всех экспериментов:")
print(f"  Baseline RF:       f1=0.5665, roc_auc=0.8380")
print(f"  Optuna RF:         f1=0.5864, roc_auc=0.8449")
print(f"  CatBoost default:  f1=0.5637, roc_auc=0.8467")
print(f"  CatBoost tuned:    f1={catboost_tuned_metrics['f1']:.4f}, roc_auc={catboost_tuned_metrics['roc_auc']:.4f}")

CatBoost tuned метрики на test:
  precision: 0.6648
  recall: 0.5096
  f1: 0.5770
  roc_auc: 0.8483

Сравнение всех экспериментов:
  Baseline RF:       f1=0.5665, roc_auc=0.8380
  Optuna RF:         f1=0.5864, roc_auc=0.8449
  CatBoost default:  f1=0.5637, roc_auc=0.8467
  CatBoost tuned:    f1=0.5770, roc_auc=0.8483


---
## Сохранение артефактов

Запускать после завершения всех экспериментов. Не зависит от MLflow.

In [101]:
import json, os

os.makedirs("./outputs", exist_ok=True)

# feature engineering
with open("./outputs/feature_names_fe_sklearn.json", "w") as f:
    json.dump(feature_names, f, indent=2)

# feature selection
with open("./outputs/sfs_selected_features.json", "w") as f:
    json.dump({"names": selected_names.tolist(), "direction": "forward"}, f, indent=2)

with open("./outputs/sfs_backward_features.json", "w") as f:
    json.dump(selected_names_backward.tolist(), f, indent=2)

with open("./outputs/rfe_features.json", "w") as f:
    json.dump(selected_names_rfe.tolist(), f, indent=2)

with open("./outputs/combined_fs_features.json", "w") as f:
    json.dump(final_features, f, indent=2)

# optuna
trials_df.to_csv("./outputs/optuna_trials.csv", index=False)
with open("./outputs/optuna_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

# catboost
importance_df.to_csv("./outputs/catboost_feature_importance.csv", index=False)
with open("./outputs/catboost_best_params.json", "w") as f:
    json.dump(best_cb_params, f, indent=2)

# final model
with open("./outputs/final_columns.json", "w") as f:
    json.dump(X_full.columns.tolist(), f, indent=2)
with open("./outputs/final_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("outputs/ сохранены:", sorted(os.listdir("./outputs")))

outputs/ сохранены: ['.gitkeep', 'catboost_best_params.json', 'catboost_feature_importance.csv', 'combined_fs_features.json', 'feature_names_fe_sklearn.json', 'final_best_params.json', 'final_columns.json', 'optuna_best_params.json', 'optuna_trials.csv', 'rfe_features.json', 'sfs_backward_features.json', 'sfs_selected_features.json']


---
# MLflow — логирование

> ⚠️ **Запускать только если MLflow-сервер работает:**
> ```
> mlflow server --host 127.0.0.1 --port 5000
> ```

> Сначала запустить **MLflow Setup** ниже, затем нужные run-ячейки.

In [29]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

EXPERIMENT_NAME = "telco_churn_prediction"
mlflow.set_experiment(EXPERIMENT_NAME)

REGISTRY_MODEL_NAME = "telco_churn_model"


In [99]:
from mlflow.models import infer_signature

input_example = X_train.head(5)
signature = infer_signature(X_train, pipeline.predict(X_train.head(5)))

/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


### Run: baseline\_random\_forest

In [100]:
with mlflow.start_run(run_name="baseline_random_forest") as run:
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", pipeline.named_steps["model"].n_estimators)
    mlflow.log_param("random_state", 42)
    mlflow.log_metrics(baseline_metrics)
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
    )
    mlflow.log_artifact("./requirements.txt")
    run_id = run.info.run_id

print(f"Run завершён. ID: {run_id}")

2026/05/07 13:33:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/6130107528c344a390ef2a6bf7f7b56e.
2026/05/07 13:33:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Run завершён. ID: 6130107528c344a390ef2a6bf7f7b56e


In [101]:
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=REGISTRY_MODEL_NAME,
)

Registered model 'telco_churn_model' already exists. Creating a new version of this model...
2026/05/07 13:33:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telco_churn_model, version 2
Created version '2' of model 'telco_churn_model'.


<ModelVersion: aliases=[], creation_timestamp=1778150034533, current_stage='None', description='', last_updated_timestamp=1778150034533, name='telco_churn_model', run_id='6130107528c344a390ef2a6bf7f7b56e', run_link='', source='/home/paul/my_proj/mlflow/mlartifacts/1/6130107528c344a390ef2a6bf7f7b56e/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='2'>

### Run: feature\_engineering\_sklearn

In [112]:
with mlflow.start_run(run_name="feature_engineering_sklearn") as run:
    mlflow.log_param("model_type", "RandomForest_FE")
    mlflow.log_param("n_estimators", pipeline_fe.named_steps["model"].n_estimators)
    mlflow.log_param("random_state", 42)
    mlflow.log_metrics(fe_metrics)
    mlflow.sklearn.log_model(
        sk_model=pipeline_fe,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
    )
    mlflow.log_artifact("./outputs/feature_names_fe_sklearn.json")
    run_id = run.info.run_id

print(f"Run завершён. ID: {run_id}")

2026/05/07 13:44:37 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_engineering_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/bce471829cb64fdaa06ad44641f96dc0.
2026/05/07 13:44:37 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Run завершён. ID: bce471829cb64fdaa06ad44641f96dc0


In [113]:
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=REGISTRY_MODEL_NAME,
)

Registered model 'telco_churn_model' already exists. Creating a new version of this model...
2026/05/07 13:45:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telco_churn_model, version 3
Created version '3' of model 'telco_churn_model'.


<ModelVersion: aliases=[], creation_timestamp=1778150719122, current_stage='None', description='', last_updated_timestamp=1778150719122, name='telco_churn_model', run_id='bce471829cb64fdaa06ad44641f96dc0', run_link='', source='/home/paul/my_proj/mlflow/mlartifacts/1/bce471829cb64fdaa06ad44641f96dc0/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='3'>

### Run: feature_selection_sfs_forward 

In [ ]:
from mlflow.models import infer_signature
import pandas as pd

# Готовим данные для сигнатуры (DataFrame с осмысленными именами)
X_train_selected_df = pd.DataFrame(X_train_selected, columns=selected_names)

signature_sfs = infer_signature(
    X_train_selected_df,
    model_sfs.predict(X_train_selected_df.head(5))
)
input_example_sfs = X_train_selected_df.head(5)

with mlflow.start_run(run_name="feature_selection_sfs_forward") as run:
    mlflow.log_param("model_type", "RandomForest_SFS_forward")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("direction", "forward")
    mlflow.log_param("n_features_total", len(feature_names_clean))
    mlflow.log_param("n_features_selected", len(selected_names))
    mlflow.log_param("scoring", "f1")
    mlflow.log_param("cv", 3)

    mlflow.log_metrics(sfs_metrics)

    mlflow.sklearn.log_model(
        sk_model=model_sfs,
        artifact_path="model",
        signature=signature_sfs,
        input_example=input_example_sfs,
    )

    mlflow.log_artifact("sfs_selected_features.json")
    mlflow.log_artifact("../requirements.txt")

    print(f"Run завершён. ID: {run.info.run_id}")

### Run: feature_selection_sfs_backward_RFE

In [31]:
import json
import pandas as pd
from mlflow.models import infer_signature

with mlflow.start_run(run_name="feature_selection_sfs_backward") as run:
    mlflow.log_param("model_type", "RandomForest_SFS_backward")
    mlflow.log_param("direction", "backward")
    mlflow.log_param("n_features_total", len(feature_names_clean))
    mlflow.log_param("n_features_selected", len(selected_names_backward))
    mlflow.log_param("scoring", "f1")
    mlflow.log_param("cv", 3)
    mlflow.log_metrics(sfs_backward_metrics)
    mlflow.log_artifact("./outputs/sfs_backward_features.json")
    mlflow.log_artifact("./requirements.txt")

with mlflow.start_run(run_name="feature_selection_rfe") as run:
    mlflow.log_param("model_type", "RandomForest_RFE")
    mlflow.log_param("n_features_total", len(feature_names_clean))
    mlflow.log_param("n_features_selected", len(selected_names_rfe))
    mlflow.log_param("step", 1)
    mlflow.log_metrics(rfe_metrics)
    mlflow.log_artifact("./outputs/rfe_features.json")

X_train_final_df = pd.DataFrame(X_train_final_fs, columns=feature_names_clean[final_mask])
signature_fs = infer_signature(X_train_final_df, model_combined_fs.predict(X_train_final_df.head(5)))
input_example_fs = X_train_final_df.head(5)

with mlflow.start_run(run_name="feature_selection_combined") as run:
    mlflow.log_param("model_type", "RandomForest_FS_combined")
    mlflow.log_param("n_features_selected", len(final_features))
    mlflow.log_param("selection_method", "intersection_of_3_methods")
    mlflow.log_metrics(combined_fs_metrics)
    mlflow.sklearn.log_model(
        sk_model=model_combined_fs,
        artifact_path="model",
        signature=signature_fs,
        input_example=input_example_fs,
    )
    mlflow.log_artifact("./outputs/combined_fs_features.json")

2026/05/11 13:20:08 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_selection_sfs_backward at: http://127.0.0.1:5000/#/experiments/1/runs/9d253a96e4704305b5ee0d6e9a238d0a.
2026/05/11 13:20:08 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
2026/05/11 13:20:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_selection_rfe at: http://127.0.0.1:5000/#/experiments/1/runs/2c27d2a83b7642bb8598e5850b302a78.
2026/05/11 13:20:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2026/05/11 13:20:16 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_selection_combined at: http://127.0.0.1:5000/#/experiments/1/runs/8c665c265ef94e8ab088bbc550c8b09e.
2026/05/11 13:20:16 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


### Run: Optuna

In [41]:
import json
from mlflow.models import infer_signature

signature_opt = infer_signature(X_train, pipeline_optuna.predict(X_train.head(5)))
input_example_opt = X_train.head(5)

with mlflow.start_run(run_name="optuna_random_forest") as run:
    mlflow.log_param("model_type", "RandomForest_optuna")
    mlflow.log_param("n_estimators", best_params["n_estimators"])
    mlflow.log_param("max_depth", best_params["max_depth"])
    mlflow.log_param("max_features", best_params["max_features"])
    mlflow.log_param("random_state", 42)
    mlflow.log_param("optuna_trials", 15)
    mlflow.log_param("optuna_sampler", "TPESampler")
    mlflow.log_param("optuna_cv", 3)
    mlflow.log_param("optuna_scoring", "f1")
    mlflow.log_param("optuna_best_cv_score", study.best_value)
    mlflow.log_metrics(optuna_metrics)
    mlflow.sklearn.log_model(
        sk_model=pipeline_optuna,
        artifact_path="model",
        signature=signature_opt,
        input_example=input_example_opt,
    )
    mlflow.log_artifact("./outputs/optuna_trials.csv")
    mlflow.log_artifact("./outputs/optuna_best_params.json")
    mlflow.log_artifact("./requirements.txt")
    run_id_optuna = run.info.run_id
    print(f"Run завершён. ID: {run_id_optuna}")

mlflow.register_model(
    model_uri=f"runs:/{run_id_optuna}/model",
    name="telco_churn_model",
)
print("Зарегистрирована новая версия модели в Registry")

/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/05/11 14:14:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run optuna_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/a974ec0741774cedb7aca409e2a5f6e7.
2026/05/11 14:14:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
Registered model 'telco_churn_model' already exists. Creating a new version of this model...
2026/05/11 14:14:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telco_churn_model, version 4


Run завершён. ID: a974ec0741774cedb7aca409e2a5f6e7
Зарегистрирована новая версия модели в Registry


Created version '4' of model 'telco_churn_model'.


### Run: CatBoost

In [48]:
from mlflow.models import infer_signature

signature_cb = infer_signature(X_train, cb_model.predict(X_train.head(5)))
input_example_cb = X_train.head(5)

with mlflow.start_run(run_name="catboost_default") as run:
    mlflow.log_param("model_type", "CatBoost")
    mlflow.log_param("iterations_set", 2000)
    mlflow.log_param("iterations_actual", cb_model.tree_count_)
    mlflow.log_param("best_iteration", cb_model.best_iteration_)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("depth", 6)
    mlflow.log_param("l2_leaf_reg", 3.0)
    mlflow.log_param("early_stopping_rounds", 50)
    mlflow.log_param("eval_metric", "F1")
    mlflow.log_param("random_seed", 42)
    mlflow.log_metrics(catboost_metrics)
    mlflow.catboost.log_model(
        cb_model=cb_model,
        artifact_path="model",
        signature=signature_cb,
        input_example=input_example_cb,
    )
    mlflow.log_artifact("./outputs/catboost_feature_importance.csv")
    mlflow.log_artifact("./requirements.txt")
    run_id_cb = run.info.run_id
    print(f"Run завершён. ID: {run_id_cb}")

mlflow.register_model(
    model_uri=f"runs:/{run_id_cb}/model",
    name="telco_churn_model",
)
print("Зарегистрирована новая версия в Registry")

/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/05/11 20:41:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run catboost_default at: http://127.0.0.1:5000/#/experiments/1/runs/0e2a558855eb465296ef2233da0ac572.
2026/05/11 20:41:15 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Run завершён. ID: 0e2a558855eb465296ef2233da0ac572


Registered model 'telco_churn_model' already exists. Creating a new version of this model...
2026/05/11 20:41:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telco_churn_model, version 5


Зарегистрирована новая версия в Registry


Created version '5' of model 'telco_churn_model'.


### Run: Catboost+Optuna

In [52]:
import json
from mlflow.models import infer_signature

signature_cb_tuned = infer_signature(X_train, cb_model_tuned.predict(X_train.head(5)))
input_example_cb_tuned = X_train.head(5)

with mlflow.start_run(run_name="catboost_optuna") as run:
    mlflow.log_param("model_type", "CatBoost_tuned")
    mlflow.log_param("learning_rate", best_cb_params["learning_rate"])
    mlflow.log_param("depth", best_cb_params["depth"])
    mlflow.log_param("l2_leaf_reg", best_cb_params["l2_leaf_reg"])
    mlflow.log_param("iterations_actual", cb_model_tuned.tree_count_)
    mlflow.log_param("best_iteration", cb_model_tuned.best_iteration_)
    mlflow.log_param("optuna_trials", 15)
    mlflow.log_param("optuna_best_eval_f1", study_cb.best_value)

    mlflow.log_metrics(catboost_tuned_metrics)

    mlflow.catboost.log_model(
        cb_model=cb_model_tuned,
        artifact_path="model",
        signature=signature_cb_tuned,
        input_example=input_example_cb_tuned,
    )

    mlflow.log_artifact("./outputs/catboost_best_params.json")
    mlflow.log_artifact("./requirements.txt")

    run_id_cb_tuned = run.info.run_id
    print(f"Run завершён. ID: {run_id_cb_tuned}")

mlflow.register_model(
    model_uri=f"runs:/{run_id_cb_tuned}/model",
    name="telco_churn_model",
)
print("Зарегистрирована новая версия в Registry")

/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/05/11 20:58:21 INFO mlflow.tracking._tracking_service.client: 🏃 View run catboost_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/97634310987f48ff9b36b129d07390f6.
2026/05/11 20:58:21 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Run завершён. ID: 97634310987f48ff9b36b129d07390f6


Registered model 'telco_churn_model' already exists. Creating a new version of this model...
2026/05/11 20:58:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telco_churn_model, version 6


Зарегистрирована новая версия в Registry


Created version '6' of model 'telco_churn_model'.


---
# Final Model

In [53]:
import pandas as pd

# train + test объединяем для финального обучения
y_train_num_series = pd.Series(y_train_num, index=X_train.index)
y_test_num_series = pd.Series(y_test_num, index=X_test.index)

X_full = pd.concat([X_train, X_test]).sort_index()
y_full_num = pd.concat([y_train_num_series, y_test_num_series]).sort_index().to_numpy()

print(f"X_full shape: {X_full.shape}")
print(f"Распределение y: {pd.Series(y_full_num).value_counts(normalize=True).to_dict()}")

X_full shape: (7043, 20)
Распределение y: {0: 0.7346301292063041, 1: 0.2653698707936959}


In [54]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=258,
        max_depth=10,
        max_features=0.26364247048639056,
        random_state=42,
        n_jobs=-1,
    )),
])

final_pipeline.fit(X_full, y_full_num)
print("Финальная модель обучена на всей выборке")

Финальная модель обучена на всей выборке


In [55]:
from mlflow.models import infer_signature
import json

signature_final = infer_signature(X_full, final_pipeline.predict(X_full.head(5)))
input_example_final = X_full.head(5)

with mlflow.start_run(run_name="production_final_random_forest") as run:
    mlflow.log_param("model_type", "RandomForest_FINAL")
    mlflow.log_param("trained_on", "full_dataset (train + test)")
    mlflow.log_param("source_run", "optuna_random_forest")
    for k, v in best_params.items():
        mlflow.log_param(k, v)
    mlflow.log_metrics({
        "precision_on_held_out_test": 0.6613,
        "recall_on_held_out_test": 0.5268,
        "f1_on_held_out_test": 0.5864,
        "roc_auc_on_held_out_test": 0.8449,
    })
    mlflow.set_tag("note", "metrics are from optuna_random_forest run (last honest test before full-fit)")
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="model",
        signature=signature_final,
        input_example=input_example_final,
    )
    mlflow.log_artifact("./outputs/final_columns.json")
    mlflow.log_artifact("./outputs/final_best_params.json")
    mlflow.log_artifact("./requirements.txt")
    run_id_final = run.info.run_id
    print(f"Финальный Run ID: {run_id_final}")

model_version = mlflow.register_model(
    model_uri=f"runs:/{run_id_final}/model",
    name="telco_churn_model",
)
print(f"Версия в реестре: {model_version.version}")

/home/paul/my_proj/.venv_my_proj/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/05/11 20:59:56 INFO mlflow.tracking._tracking_service.client: 🏃 View run production_final_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/57444a75c2c840238e40d2dde43af954.
2026/05/11 20:59:56 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
Registered model 'telco_churn_model' already exists. Creating a new version of this model...
2026/05/11 20:59:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: telco_churn_model, version 7


Финальный Run ID: 57444a75c2c840238e40d2dde43af954
Версия в реестре: 7


Created version '7' of model 'telco_churn_model'.


In [56]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="telco_churn_model",
    alias="production",
    version=model_version.version,
)
client.set_model_version_tag(
    name="telco_churn_model",
    version=model_version.version,
    key="stage",
    value="Production",
)

print(f"Версия {model_version.version} → production alias")
print(f"models:/telco_churn_model@production")

Версия 7 помечена как Production
Доступ через: models:/telco_churn_model@production
